# STAC >> fs

In [1]:
import sys
sys.path.insert(1, '../utils/')

In [2]:
# reload module before executing code
%load_ext autoreload
%autoreload 2
%matplotlib inline

# # Standard library imports
import os
# import re
import time
# from copy import deepcopy
# # from urllib.parse import urlparse
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # Third-party library imports
# import requests
# import pystac_client
# import planetary_computer as pc
    
from IPython.display import JSON
from pathlib import Path
# from tqdm import tqdm

# Local module imports
from utils.nostra_add_product import (
    select_and_save_product,
    add_product_via_cli,
    parse_product_yaml
)
from utils.nostra_mapping import display_crosshair, MapHandler

from utils.nostra_stac_to_filesystem import (
    stac_to_filesystem,
    list_filesystem_tree,
    find_last_level_folders,
    prepare_filesystem_folders,
    dc_add_dataset,
)
# from utils.nostra_minio import (
#     connect_and_check_endpoint,
#     create_bucket,
#     set_anonymous_download_permissions,
#     list_bucket_tree,
#     list_last_level_folders,
#     empty_bucket,
#     delete_bucket,
# )
# from utils.nostra_stac_to_minio import (
#     stac_to_minio,
#     valid_minio_folders,
#     prepare_minio_folders,
#     index_minio_yamls,
# )
from utils.nostra_tools import extract_path_from_string, style_output_cells

## 1. Configure and add a new product

In [3]:
# Configuration

yamls_dict = {
        "local": "/conf",
        "dc_tools": "https://raw.githubusercontent.com/opendatacube/odc-tools/refs/heads/develop/apps/dc_tools/tests/data/example_product_list.csv"
    }

Before indexing, a product description in YAML format needs to be added to the datacube.

The following cell generates a YAML template based on a selection of existing products. You will need to adapt this template to the product you want to add before proceeding with the subsequent cells.

In [4]:
select_and_save_product(yamls_dict, "example.yaml")

In [5]:
# adapted yaml

yaml_file = 'ls89_c2l2_sp_local-product.yaml'
parse_product_yaml(yaml_file)

YAML file is valid!
Product name: ls89_c2l2_sp_local
Description: USGS Landsat 8 and 9 Collection 2 Level-2 Science Products, consisting of atmospherically corrected surface reflectance and surface temperature image data.
Metadata type: eo3
Number of measurements: 3
Measurements:
  1. blue
  2. green
  3. red


In [6]:
# TbD: Print variable and values to be found on yaml

r = add_product_via_cli(yaml_file, update=True)

Failed to add product: 2026-01-20 08:21:32,641 501 datacube-product ERROR 'accessories' does not match any of the regexes: 'default_\\w+'

Failed validating 'additionalProperties' in schema:
    {'$schema': 'https://json-schema.org/draft/2020-12/schema',
     'description': 'Schema for dataset types.',
     'type': 'object',
     'properties': {'name': {'type': 'string', 'pattern': '^\\w+$'},
                    'description': {'type': 'string'},
                    'metadata_type': {'oneOf': [{'type': 'string'},
                                                {'$ref': 'metadata-type-schema.yaml'}]},
                    'license': {'title': 'Product license',
                                'description': 'Either a SPDX License '
                                               "identifier, 'various' or "
                                               "'proprietary'",
                                'type': 'string',
                                'pattern': '^[\\w\\-.+]+$'},
          

## 2. Identify, download and transfer scenes files on MinIO

In [7]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, dc = map_handler.create_map()
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

Map(center=[0, 0], controls=(AttributionControl(options=['position', 'prefix'], position='bottomright'), Searc…

<IPython.core.display.Javascript object>

In [ ]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple
if aoi_poly is None:
    style_output_cells('salmon', border_color='red', border_width='2px')
    print('The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.')
else:
    style_output_cells()

In [ ]:
# Configuration

stac_endpoint = "https://planetarycomputer.microsoft.com/api/stac/v1"
stac_collection = "landsat-c2-l2"
stac_platforms=["landsat-8", "landsat-9"]
stac_time = ('2024-07-15', '2024-08-01')  # end date is not inclusive !
stac_layers = ['red', 'green', 'blue'] # all if empty ['atran', 'blue', 'cdist', 'coastal', 'drad', 'emis', 'emsd', 'green', 'lwir11', 'nir08', 'qa', 'qa_aerosol', 'qa_pixel', 'qa_radsat', 'red', 'swir16', 'swir22', 'trad', 'urad']

# product_type = 'Landsat'
product_name = 'ls89_c2l2_sp_local'  # 'landsat_ot_c2_l2'
yaml_conf = 'landsat89_mtl_config.yaml'

fs_path = '/local_data'

In [ ]:
%%time

# Download from STAC to filesystem

assets_dict, download_results = stac_to_filesystem(
    base_path = fs_path,
    stac_endpoint = stac_endpoint, stac_collection = stac_collection,
    platforms=stac_platforms,
    stac_time = stac_time, stac_layers = stac_layers, t1_only = True,
    aoi_poly = aoi_poly,
    overwrite=True, complete=True, dry_run=False, verbose=False
)
JSON(download_results) # display summary

In [ ]:
# Optional

list_filesystem_tree(fs_path, 'landsat-c2/level-2/standard/oli-tirs',
                 max_recursion_level=4, show_sizes=True)

## Prepare yamls now !

In [ ]:
# List scenes to process

# OPTION 1: from stac_to_filesystem output
folders = list(assets_dict.keys())

# # OPTION 2: all folders on filesystem
# folders = find_last_level_folders(fs_path)  # "*/oli-tirs/**/*2023072*"

# recreate absolute path
folders = [os.path.join(fs_path, folder) for folder in folders]
folders

In [ ]:
# optionally remove existing yamls

for folder_path in folders:
    folder = Path(folder_path)
    if folder.is_dir():
        for yaml_file in folder.glob("*.yaml"):
            yaml_file.unlink()  # This deletes the file
            print(f"Deleted: {yaml_file}")

In [ ]:
folders

In [ ]:
results = prepare_filesystem_folders(folders, product_name, yaml_conf)

In [ ]:
results

## Index scenes

In [ ]:
yamls =[extract_path_from_string(r['result']) for r in results]
yamls

In [ ]:
newly_added, already_indexed, failed = dc_add_dataset(yamls, max_workers=4)
    
print(f"\n{'='*60}")
print(f"Summary:")
print(f"  Newly added: {len(newly_added)}")
print(f"  Already indexed: {len(already_indexed)}")
print(f"  Failed: {len(failed)}")
print(f"  Total: {len(yamls)}")
print(f"{'='*60}")

You can test data were properly indexed by using [Test_fs_indexation.ipynb](Test_fs_indexation.ipynb).